In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline

In [4]:
data = pd.read_csv("pima-indians-diabetes.csv", header = None)
data.columns = ['var_preg', 'var_plas', 'var_pres', 'var_skin', 'var_test', 'var_mass', 'var_pedi', 'var_age', 'var_class']
print(data.head())

   var_preg  var_plas  var_pres  var_skin  var_test  var_mass  var_pedi  \
0         6       148        72        35         0      33.6     0.627   
1         1        85        66        29         0      26.6     0.351   
2         8       183        64         0         0      23.3     0.672   
3         1        89        66        23        94      28.1     0.167   
4         0       137        40        35       168      43.1     2.288   

   var_age  var_class  
0       50          1  
1       31          0  
2       32          1  
3       21          0  
4       33          1  


In [18]:
# print counts() to check empty fields
print("data shape", data.shape)
print("data count", data.count(), "\n") # no empty fields

print("Is nulls", pd.isnull(data)) # prints True for every value if np.nan
print("Any null valiue?", data.isnull().values.any())  # no empty fields

data shape (768, 9)
data count var_preg     768
var_plas     768
var_pres     768
var_skin     768
var_test     768
var_mass     768
var_pedi     768
var_age      768
var_class    768
dtype: int64 

Is nulls      var_preg  var_plas  var_pres  var_skin  var_test  var_mass  var_pedi  \
0       False     False     False     False     False     False     False   
1       False     False     False     False     False     False     False   
2       False     False     False     False     False     False     False   
3       False     False     False     False     False     False     False   
4       False     False     False     False     False     False     False   
..        ...       ...       ...       ...       ...       ...       ...   
763     False     False     False     False     False     False     False   
764     False     False     False     False     False     False     False   
765     False     False     False     False     False     False     False   
766     False     Fals

In [20]:
X = data.drop(['var_class'], axis = 1)
Y = data['var_class']


In [21]:
# 1. Define a consistent random state for reproducibility
SEED = 7

# --- 2. Feature Preprocessing (Scaling and FeatureUnion) ---
# The pipeline is restructured to include a scaling step *before* the FeatureUnion.
preprocessors = [
    # Step 1: Standardize features (mean=0, std=1)
    ('scaler', StandardScaler()), 
    
    # Step 2: Parallel feature engineering on the scaled data
    ('feature_union', FeatureUnion([
        ('pca', PCA(n_components=3, random_state=SEED)),
        ('select_best', SelectKBest(k=6))
        # Note: SelectKBest typically uses statistical tests like chi2 (default), 
        # which can be sensitive to scaling, but it's often more robust 
        # to apply it after scaling than to leave out scaling entirely.
    ]))
]

# Create a pipeline just for preprocessing
preprocessing_pipe = Pipeline(preprocessors)



(768, 8)

In [ ]:
# --- 3. Full Model Pipeline ---
# This pipeline chains the preprocessing steps and the final model
model = Pipeline([
    ('preprocessor', preprocessing_pipe), # First step: All feature processing
    # Second step: The final classifier
    ('logistic', LogisticRegression(random_state=SEED, solver='liblinear')) 
    # Added solver='liblinear' for smaller datasets/consistency
])


In [ ]:
# --- 4. Cross-Validation Setup (Improved KFold) ---
# KFold with shuffle=True and random_state for reproducibility
varkfold = KFold(n_splits=10, shuffle=True, random_state=SEED)



In [ ]:
# --- 5. The Pipeline is tested here ---
# varX and varY must be defined in the context where this code runs
# Example placeholder for varX and varY:
# varX = np.random.rand(100, 20)
# varY = np.random.randint(0, 2, 100) 
try:
    dataresults = cross_val_score(model, varX, varY, cv=varkfold, scoring='accuracy')
    print(f"Cross-Validation Mean Accuracy: {dataresults.mean():.4f}")
except NameError:
    print("Please ensure 'varX' and 'varY' (your data and labels) are defined before running.")